# Local Carry Laws

The local carry recurrence is

$$r_{k+1} = \left\lfloor \frac{c_k + r_k}{B} \right\rfloor.$$

Therefore carry is active exactly when

$$c_k + r_k \ge B.$$

The incoming carry matters: `raw_c` alone is not enough to decide activation.

In [ ]:
from pathlib import Path
import sys

sys.path.append("../src")
if (Path.cwd() / "src").exists():
    sys.path.append(str(Path.cwd() / "src"))
elif (Path.cwd().parent / "src").exists():
    sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from experiments import experiment_local_carry_laws
from local_carry_laws import check_local_activation_law

In [ ]:
RESULTS = Path("results") if (Path.cwd() / "results").exists() else Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS / "local_carry_laws.csv"

if csv_path.exists():
    data = pd.read_csv(csv_path)
else:
    data = experiment_local_carry_laws(
        n_trials=20,
        num_digits=10,
        B=10,
        densities=[0.2, 0.5, 0.7, 1.0],
        output_csv=str(csv_path),
    )

data.head()

In [ ]:
law_ok = check_local_activation_law(data)
print("Local activation law passed:", law_ok)
print("Rows:", len(data))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].scatter(data["raw_c"], data["outgoing_carry"], alpha=0.35)
axes[0, 0].set_xlabel("raw_c")
axes[0, 0].set_ylabel("outgoing_carry")
axes[0, 0].set_title("raw_c vs outgoing carry")

axes[0, 1].scatter(data["total_s"], data["outgoing_carry"], alpha=0.35, color="#E45756")
axes[0, 1].axvline(data["B"].iloc[0], color="black", linestyle="--", linewidth=1)
axes[0, 1].set_xlabel("total_s = raw_c + incoming_carry")
axes[0, 1].set_ylabel("outgoing_carry")
axes[0, 1].set_title("total_s vs outgoing carry")

density_rate = data.groupby("density")["carry_active"].mean()
axes[1, 0].plot(density_rate.index, density_rate.values, marker="o")
axes[1, 0].set_xlabel("density")
axes[1, 0].set_ylabel("carry_active rate")
axes[1, 0].set_title("density vs carry-active rate")

k_rate = data.groupby("k")["carry_active"].mean()
axes[1, 1].bar(k_rate.index, k_rate.values, color="#4C78A8")
axes[1, 1].set_xlabel("degree k")
axes[1, 1].set_ylabel("carry_active rate")
axes[1, 1].set_title("degree vs carry-active rate")

for ax in axes.ravel():
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

The plot using `total_s` shows the threshold law directly. The plot using `raw_c` is noisier because carry activation also depends on `incoming_carry`. This is the local-to-global feature of carry: each local outgoing carry changes the next local condition.